# Approach 1
This approach applies TCDB to acquire transporters and their TC identification (TCID). Subsequently, mechanisms for the relevant families are obtained manually, before they are connected to the reaction of the mechanism. This will in turn be crosschecked with Rhea, which is mapped to through UniProt IDs (UID).

Semantically, it will follow something along these lines: TCID + substrate + mechanism -> chemical reaction. Connect this and compare with reaction on Rhea.

In [2]:
import requests
from Bio import SeqIO
from io import StringIO
import pandas as pd
import numpy as np

The relevant files from the Mapping Files on TCDB, are as following:
1) "Tab-delimited table mapping TC uniprot/refseq accessions to TC systems" -tc_uid_url
2) "Tab-delimited table mapping TC systems to their substrates and ChEBI IDs" - tc_substrates_url

In [3]:
def fetch_data(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text


def parse_data(uid_txt, substrates_txt):

    # TCID and Accesion ID (UID/RefSeq)
    uid_data = [line.split("\t") for line in uid_txt.strip().split("\n")]
    df_uid = pd.DataFrame(uid_data, columns=["UID", "TCID"])

    # Substrates (TCID, CHEBI ID and CHEBI Name)
    substrates_lines = substrates_txt.strip().split("\n")
    substrate_data = [[line.split("\t")[0], chebi.split(";")[0], chebi.split(";")[1]]
        for line in substrates_lines
        for chebi in line.split("\t")[1].split("|")]
    
    df_substrates = pd.DataFrame(substrate_data, columns=["TCID", "CHEBI ID", "CHEBI Name"])

    return df_uid, df_substrates

In [4]:
tc_uid_url = "https://www.tcdb.org/cgi-bin/projectv/public/acc2tcid.py"
tc_substrates_url = "https://www.tcdb.org/cgi-bin/substrates/getSubstrates.py"

tc_uid_txt = fetch_data(tc_uid_url)
tc_substrates_txt = fetch_data(tc_substrates_url)

df_uid, df_substrates = parse_data(tc_uid_txt, tc_substrates_txt)

First I'll merge the DFs.\
The relevant subclasses were retrieved in Misc/TCDB_composition.ipynb, and are: [1.A, 1.B, 1.C, 2.A, 3.A]\
From these subclasses, the ten most populated familes were obtained for further analysis. These, alongside their general mechanism and acting entity can be obtained from Misc/All_comp/family_mechanisms_entity_all.tsv.\
Only the top ten families in each of the subclasses above are of interest, but the rest will not be removed before the end.

In [13]:
df = pd.merge(df_uid, df_substrates, on="TCID", how="left")


# Importing the families and related mechanisms
df_family_mechanisms = pd.read_csv("../Misc/All_comp/families_mechanisms_entity_final.tsv", sep="\t")
df_family_mechanisms["Mechanism"] = df_family_mechanisms["Mechanism"].replace({"â‡Œ": "⇌", "â†’": "→"}, regex=True)

# Filter out families not in top 10 of each of the selected subclasses
df["Family"] = df["TCID"].apply(lambda x: ".".join(x.split(".")[:3]))
df = df.merge(df_family_mechanisms[["Family", "Mechanism", "Acting Entity"]], on="Family", how="left")
df
# Keeping the "Family"-column until only top ten families of each chosen subclass is removed

,UID,TCID,CHEBI ID,CHEBI Name,Family,Mechanism,Acting Entity
0,A0CIB0,1.A.17.1.13,CHEBI:3731,chloride,1.A.17,Cl- (out) ⇌ Cl- (in),Cl-
1,A0CIB0,1.A.17.1.13,CHEBI:3731,chloride,1.A.17,Cations (out) ⇌ Cations (in),Cations
2,A0CS82,9.B.82.1.5,NaN,NaN,9.B.82,NaN,NaN
3,A0CX44,1.A.3.2.4,CHEBI:3308,calcium(2+),1.A.3,NaN,NaN
4,A0D5K0,2.A.66.3.4,CHEBI:8150,phospholipid,2.A.66,NaN,NaN
...,...,...,...,...,...,...,...
39839,KEG13462.1,3.A.2.2.11,CHEBI:5584,hydron,3.A.2,nH+/Na+ (in) + ATP ⇌ nH+/Na+ (out) + ADP + Pi,nH+/Na+
39840,KEG13226.1,3.A.2.2.11,CHEBI:5584,hydron,3.A.2,nH+/Na+ (in) + ATP ⇌ nH+/Na+ (out) + ADP + Pi,nH+/Na+
39841,B2BNE9,1.A.60.1.4,CHEBI:25367,molecule,1.A.60,NaN,NaN
39842,AAL11723.1,1.F.3.1.6,CHEBI:14911,protein,1.F.3,NaN,NaN


Now, many of the CHEBI IDs are secondary IDs, and needs to be converted in order to map to Rhea for cross-checking.

In [14]:
df_s2p = pd.read_csv("../ChEBI/s2p.tsv", sep="\t")
secondary_to_primary = dict(zip(df_s2p["Secondary_ID"], df_s2p["Primary_ID"]))
df["CHEBI ID"] = df["CHEBI ID"].apply(lambda x: secondary_to_primary.get(x, x))

Future plan: Go through families_mechanisms_all.tsv and create another column for the acting entity in the reaction. The acting entity x will be marked as such: {x}\
This was first conudcted through AI, as this is a tedious manual task, before it was verified and edited by hand.

In [15]:
def create_reaction_row(row):

    if pd.isna(row["Acting Entity"]) or pd.isna(row["CHEBI Name"]):
        return row["Mechanism"]
    

    mechanisms = row["Mechanism"].split(", ")
    acting_entities = str(row["Acting Entity"]).split(", ")
    chebi_name = str(row["CHEBI Name"])

    reactions = []
    for mechanism, entity in zip(mechanisms, acting_entities):
        reaction = mechanism.replace(entity, chebi_name)
        reactions.append(reaction)
    
    return ", ".join(reactions)

df["Reaction"] = df.apply(create_reaction_row, axis=1)

Now the next focus will be to obtain the Rhea reactions, including both the names and the ChEBI IDs. This will go through UniProt.\
Starting off with mapping the correct Rhea IDs (RID) through a UniProt SPARQLE-query, saved as UniProt/Modified_queries/UID_RID.tsv.

In [16]:
uid_rhea_map = pd.read_csv("../UniProt/Modified_queries/RID_UID.tsv", sep="\t")

df = df.merge(uid_rhea_map, left_on="UID", right_on="UID", how="left")
df["RID"] = df["RID"].apply(lambda x: f"RHEA:{int(x)}" if pd.notnull(x) else x)

The following step is to include the mapped reaction data for each RID. Both the equation, the ChEBI IDs and the ChEBI Name. This is stored in the columns titled as such: R:{col_name}\
The Rhea data is obtained from rhea-db.org 27.01.25, and can be found in the Rhea-folder.

In [17]:
rhea = pd.read_csv("../Rhea/Rhea.tsv", sep="\t")

rhea["Reaction identifier"] = rhea["Reaction identifier"].astype(str)
df["RID"] = df["RID"].astype(str)
df = df.merge(rhea, left_on="RID", right_on="Reaction identifier", how="left")

df = df.drop(columns=["Reaction identifier"])
df = df.rename(columns=lambda x: f"R:{x}" if x not in
               ["TCID", "UID", "CHEBI ID", "CHEBI Name", "Mechanism", "Acting Entity", "Reaction", "RID", "Family"]
               else x)
df["RID"] = df["RID"].replace("nan", np.nan)

df

,UID,TCID,CHEBI ID,CHEBI Name,Family,Mechanism,Acting Entity,Reaction,RID,R:Equation,R:ChEBI name,R:ChEBI identifier,R:EC number
0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,1.A.17,Cl- (out) ⇌ Cl- (in),Cl-,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN,NaN
1,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,1.A.17,Cations (out) ⇌ Cations (in),Cations,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN,NaN
2,A0CS82,9.B.82.1.5,NaN,NaN,9.B.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0CX44,1.A.3.2.4,CHEBI:29108,calcium(2+),1.A.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0D5K0,2.A.66.3.4,CHEBI:8150,phospholipid,2.A.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
56999,KEG13462.1,3.A.2.2.11,CHEBI:15378,hydron,3.A.2,nH+/Na+ (in) + ATP ⇌ nH+/Na+ (out) + ADP + Pi,nH+/Na+,hydron (in) + ATP ⇌ hydron (out) + ADP + Pi,NaN,NaN,NaN,NaN,NaN
57000,KEG13226.1,3.A.2.2.11,CHEBI:15378,hydron,3.A.2,nH+/Na+ (in) + ATP ⇌ nH+/Na+ (out) + ADP + Pi,nH+/Na+,hydron (in) + ATP ⇌ hydron (out) + ADP + Pi,NaN,NaN,NaN,NaN,NaN
57001,B2BNE9,1.A.60.1.4,CHEBI:25367,molecule,1.A.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57002,AAL11723.1,1.F.3.1.6,CHEBI:14911,protein,1.F.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


VERY many of the UIDs are refseq IDs (RSIDs), and hence unattainable through UniProt-Rhea mapping. Therefore, a query was written, and can be found in UniProt/UniProt_Rhea.ipynb as query2_new. This obtains the UID and RSID whenever attainable. The query was run online on https://sparql.uniprot.org/, and the result is stored in UniProt/Modified_queries/query2_new.csv. After a second of thought, I realize that this file is too large to push to Git, so this must be created manually. The notebook modifies the csv as wanted, easing the use here. The same goes for the modified file, UID_RSID.tsv is close to the max push size, hence not included in the repo.

Now, a new column is created AID (Accession ID), in order to retain the elements of the UID-column tht does not have an RID. Following, these will be attempted converted into UIDs, assuming many of them are RSIDs. Then an RID will be attributed.

In [20]:
refseq = pd.read_csv("../UniProt/Modified_queries/UID_RSID.tsv", sep="\t")

df["AID"] = df["UID"].where(df["RID"].isna())
df = df.merge(refseq, left_on="AID", right_on="RSID", how="left")
df = df.rename(columns={"UID_x": "UID", "UID_y": "UID2"})
df = df.drop(columns=["RSID"])

df

,UID,TCID,CHEBI ID,CHEBI Name,Family,Mechanism,Acting Entity,Reaction,RID,R:Equation,R:ChEBI name,R:ChEBI identifier,R:EC number,AID,UID2
0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,1.A.17,Cl- (out) ⇌ Cl- (in),Cl-,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN,NaN,A0CIB0,NaN
1,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,1.A.17,Cations (out) ⇌ Cations (in),Cations,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN,NaN,A0CIB0,NaN
2,A0CS82,9.B.82.1.5,NaN,NaN,9.B.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A0CS82,NaN
3,A0CX44,1.A.3.2.4,CHEBI:29108,calcium(2+),1.A.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A0CX44,NaN
4,A0D5K0,2.A.66.3.4,CHEBI:8150,phospholipid,2.A.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A0D5K0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57052,B2BNE9,1.A.60.1.4,CHEBI:25367,molecule,1.A.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B2BNE9,NaN
57053,AAL11723.1,1.F.3.1.6,CHEBI:14911,protein,1.F.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AAL11723.1,NaN
57054,NP_001287972.1,8.A.23.1.68,NaN,NaN,8.A.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NP_001287972.1,Q9BY67
57055,NP_001287972.1,8.A.23.1.68,NaN,NaN,8.A.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NP_001287972.1,A0A4Z1


Now mapping UID2-values to RIDs. This will fill up a lot more reactions. Hopefully... Currently it is wrong. It is some errors with the name of columns etc. But this will be sorted out, and then, new RIDs, and correspondning data can be attributed in the R:{cols}.

In [21]:
rhea["Reaction identifier"] = rhea["Reaction identifier"].astype(str)
df["RID"] = df["RID"].astype(str)

df = df.merge(rhea, left_on="UID2", right_on="Reaction identifier", how="left")

df = df.drop(columns=["Reaction identifier"])

df = df.rename(columns=lambda x: f"R:{x}" if x not in
               ["TCID", "UID", "UID2", "AID", "CHEBI ID", "CHEBI Name", "Mechanism", 
                "Acting Entity", "Reaction", "RID", "Family"]
               else x)

df["RID"] = df["RID"].replace("nan", np.nan)
df

,UID,TCID,CHEBI ID,CHEBI Name,Family,Mechanism,Acting Entity,Reaction,RID,R:R:Equation,R:R:ChEBI name,R:R:ChEBI identifier,R:R:EC number,AID,UID2,R:Equation,R:ChEBI name,R:ChEBI identifier,R:EC number
0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,1.A.17,Cl- (out) ⇌ Cl- (in),Cl-,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN,NaN,A0CIB0,NaN,NaN,NaN,NaN,NaN
1,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,1.A.17,Cations (out) ⇌ Cations (in),Cations,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN,NaN,A0CIB0,NaN,NaN,NaN,NaN,NaN
2,A0CS82,9.B.82.1.5,NaN,NaN,9.B.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A0CS82,NaN,NaN,NaN,NaN,NaN
3,A0CX44,1.A.3.2.4,CHEBI:29108,calcium(2+),1.A.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A0CX44,NaN,NaN,NaN,NaN,NaN
4,A0D5K0,2.A.66.3.4,CHEBI:8150,phospholipid,2.A.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A0D5K0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57052,B2BNE9,1.A.60.1.4,CHEBI:25367,molecule,1.A.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B2BNE9,NaN,NaN,NaN,NaN,NaN
57053,AAL11723.1,1.F.3.1.6,CHEBI:14911,protein,1.F.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AAL11723.1,NaN,NaN,NaN,NaN,NaN
57054,NP_001287972.1,8.A.23.1.68,NaN,NaN,8.A.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NP_001287972.1,Q9BY67,NaN,NaN,NaN,NaN
57055,NP_001287972.1,8.A.23.1.68,NaN,NaN,8.A.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NP_001287972.1,A0A4Z1,NaN,NaN,NaN,NaN


More coding above is required....\
Finally, the removal of the instances of transporters not in top 10 of each of the selected subclasses.

In [12]:
df = df[df["Family"].isin(df_family_mechanisms["Family"])]
df = df.drop(columns=["Family"])
df

,UID,TCID,CHEBI ID,CHEBI Name,Mechanism,Acting Entity,Reaction,RID,R:Equation,R:ChEBI name,R:ChEBI identifier,R:EC number,AID,RSID
0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,Cl- (out) ⇌ Cl- (in),Cl-,chloride (out) ⇌ chloride (in),nan,NaN,NaN,NaN,NaN,NaN,XP_001437924.1
1,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,Cations (out) ⇌ Cations (in),Cations,chloride (out) ⇌ chloride (in),nan,NaN,NaN,NaN,NaN,NaN,XP_001437924.1
4,A0D5K0,2.A.66.3.4,CHEBI:8150,phospholipid,NaN,NaN,NaN,nan,NaN,NaN,NaN,NaN,NaN,XP_001445714.1
6,A0ECD9,3.A.1.207.1,NaN,NaN,Solute (in) + ATP → Solute (out) + ADP + Pi,Solute,Solute (in) + ATP → Solute (out) + ADP + Pi,nan,NaN,NaN,NaN,NaN,NaN,XP_001460353.1
7,A0ECD9,3.A.1.207.1,NaN,NaN,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,Substrate (out) + ATP → Substrate (in) + ADP + Pi,nan,NaN,NaN,NaN,NaN,NaN,XP_001460353.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104277,WP_332273826.1,3.A.1.12.18,CHEBI:25728,osmolyte,Solute (in) + ATP → Solute (out) + ADP + Pi,Solute,osmolyte (in) + ATP → osmolyte (out) + ADP + Pi,nan,NaN,NaN,NaN,NaN,NaN,NaN
104278,WP_332273826.1,3.A.1.12.18,CHEBI:25728,osmolyte,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,osmolyte (out) + ATP → osmolyte (in) + ADP + Pi,nan,NaN,NaN,NaN,NaN,NaN,NaN
104284,KEG12320.1,3.A.2.2.11,CHEBI:15378,hydron,nH+/Na+ (in) + ATP ⇌ nH+/Na+ (out) + ADP + Pi,nH+/Na+,hydron (in) + ATP ⇌ hydron (out) + ADP + Pi,nan,NaN,NaN,NaN,NaN,NaN,NaN
104285,KEG13462.1,3.A.2.2.11,CHEBI:15378,hydron,nH+/Na+ (in) + ATP ⇌ nH+/Na+ (out) + ADP + Pi,nH+/Na+,hydron (in) + ATP ⇌ hydron (out) + ADP + Pi,nan,NaN,NaN,NaN,NaN,NaN,NaN
